# Lesson 11 | Why not scan every synapse after every spike?

The queue now contains a `source_id`. The system must determine which targets that source connects to. Today asks:
> **How can we visit only connections that actually exist instead of checking every possible neuron pair?**

Primary concept: **sparse graph representation**.


## 1. Concept ledger

**Known:** a spike event carries source_id; a queue preserves event order.

**New:** sparse graph, adjacency list, and the core idea of **Compressed Sparse Row (CSR)**.

**Preview:** next lesson connects lookup, weighted events, and target update. DDR is still later.


## 2. First view the neural network as a graph

A **graph** here has two basic pieces:

- neurons are nodes;
- synapses are directed edges.

If four neurons have only four synapses, connectivity is sparse. A 4×4 dense matrix has 16 positions even though most positions mean “no edge.”


## 3. Adjacency list: store only real edges

An **adjacency list** directly lists the real targets for each source.

Example:

- source 0 → `(1,+2)`, `(3,-1)`
- source 1 → none
- source 2 → `(1,+3)`
- source 3 → `(2,+1)`

A source-0 spike needs only two records instead of rechecking all possible target pairs.


## 4. CSR core idea: contiguous records plus a source index

**Compressed Sparse Row (CSR)** stores the nonzero entries of each row contiguously and uses an index to locate each row.

For this project, a source neuron plays the role of a row. The MDD currently describes `source_index = (start_offset, fanout_count)` plus contiguous `synapse_records`, which follows the same core idea.

```mermaid
flowchart LR
 S["source_id"] --> IDX["source_index: start,count"]
 IDX --> REC["contiguous synapse_records"]
 REC --> O["target, weight ..."]
```


## 5. Run: compress four edges into contiguous records

Predict the count for source 1, then run.


In [ ]:
edges = [
    (0, 1, 2),
    (0, 3, -1),
    (2, 1, 3),
    (3, 2, 1),
]
num_sources = 4

records = []
source_index = []
for source in range(num_sources):
    start = len(records)
    for src, target, weight in edges:
        if src == source:
            records.append((target, weight))
    source_index.append((start, len(records) - start))

print('source_index =', source_index)
print('synapse_records =', records)
for source, (start, count) in enumerate(source_index):
    print(f'source {source}:', records[start:start+count])


## 6. Observe

The result should show source 0 covering the first two records, source 1 with count 0, sources 2 and 3 with one record each, and every real edge stored once.

That is the central property behind T-009 “source index → exact synapse range.”


## 7. Try It: compare scan counts

For 1000 neurons, suppose one source has only 3 downstream synapses.

- How many possible targets does a dense scan inspect?
- Once its sparse range is known, how many synapse records need to be read?

Compare record counts only; memory latency, cache, and DDR bandwidth come later.


## 8. Homework

Complete `exercises/lesson11_sparse_graph.py`:

- `build_source_index(...)` packs edges by source into contiguous records.
- `lookup_source(...)` returns exactly that source's slice.

```bash
uv run pytest exercises/checks/check_lesson11.py -q
```


## 9. AI Task

Give AI five sources and a small edge list and ask it to produce `source_index + records`. Require it to reconstruct every source slice to prove there is no off-by-one error.


## 10. Human Check

Without AI, explain why dense scanning wastes work, why adjacency lists fit sparse connectivity, what `start_offset` and `fanout_count` mean, and why a source with count 0 is still a valid index entry.


## 11. Engineering Handoff

This lesson builds data-layout intuition for `MOD-006 source_index_store`, `MOD-007 synapse_reader`, and T-009. It does not freeze the binary image, record width, DDR layout, or formal streaming interface.


## 12. Project Trace

- Lesson: `LSN-011`
- Mapping: `RMD-008`
- Module context: `MOD-006 / MOD-007`
- Test context: `T-009`


## 13. Exit Ticket

Given a source_id and `(start,count)`, you can identify exactly which record slice to read and explain why this fits sparse connectivity better than scanning all possible synapses.
